# Credit Curve Bootstrapping

A **credit curve** maps each future date to the survival probability $Q(t) = \mathbb{P}(\tau > t)$ of a reference name, where $\tau$ is the random default time. Bootstrapping recovers a piecewise representation of $Q(t)$ from observed CDS spreads, solving for one pillar at a time (or all simultaneously) so that each market quote re-prices to par on the calibrated curve.

This notebook walks through every element of the process:

1. The **deterministic-intensity model** that anchors all the math.
2. The **CDS leg decomposition** — premium (with accrual on default) and protection (with recovery on default).
3. **`CdsQuote`** — what a market quote bundles and how it generates its schedule.
4. **`CreditCurve`** — three interpolation variables (`SURVIVAL_PROBABILITY`, `DEFAULT_SPREAD`, `FORWARD_DEFAULT_SPREAD`) and what each one parameterises.
5. **`SingleNameCDS`** — the pricer the bootstrapper uses inside its objective.
6. **`CreditCurveBootstrapper`** — sequential vs. global Newton-Raphson, and why both modes solve the same root system.
7. **Round-trip verification** and side-by-side curve plots.

Throughout, we use a fixed reference date of 2 January 2024 and a flat USD discount curve to keep the focus on the credit dimension.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd().parent if Path.cwd().name == 'examples' else Path.cwd()
sys.path.insert(0, str(root))

from database import set_db_path
set_db_path(str(root / 'examples' / 'demo.db'))

In [ ]:
import sqlite3
from database import get_db_path
from scripts.initialise import init_db, _seed_holidays

init_db()
with sqlite3.connect(get_db_path()) as conn:
    count = conn.execute("SELECT COUNT(*) FROM holidays").fetchone()[0]
    if count == 0:
        _seed_holidays(conn)
        print("Seeded demo.db with holiday data.")
    else:
        print(f"demo.db already seeded ({count} holidays). Skipping.")

In [ ]:
import math
from datetime import date

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from market_conventions import BusinessDayConvention, DayCountConvention, StubType
from market_structures.rates.curve import ZeroCurve
from schedules import CalendarType, Frequency
from schedules.day_count import day_count_fraction

from credit import (
    BootstrapMode,
    CdsQuote,
    CdsSide,
    CreditCurve,
    CreditCurveBootstrapper,
    InterpolationVariable,
    SingleNameCDS,
)

REF    = date(2024, 1, 2)
USD    = CalendarType.USD
MF     = BusinessDayConvention.FOLLOWING
ACT360 = DayCountConvention.ACT_360
ACT365 = DayCountConvention.ACT_365_FIXED

RECOVERY = 0.40

print(f"Reference date: {REF}")
print(f"Recovery rate (constant): {RECOVERY:.0%}")

## 1. The Deterministic-Intensity Model

We model the default time $\tau$ as the first jump of an inhomogeneous Poisson process with deterministic intensity $\lambda(t)$. This gives three core relations that the entire credit-curve construction is built on:

$$
Q(t) \;=\; \mathbb{P}(\tau > t) \;=\; \exp\!\Big(\!-\!\int_0^t \lambda(u)\,du\Big)
$$

$$
f_\tau(t)\,dt \;=\; -\,dQ(t) \;=\; \lambda(t)\,Q(t)\,dt
$$

$$
\text{cumulative spread } s(t) \;=\; -\frac{\ln Q(t)}{t} \;=\; \frac{1}{t}\!\int_0^t \lambda(u)\,du
$$

Three natural ways to parameterise the curve correspond to interpolating one of three quantities between pillar dates:

| Variable | Stored values | Interpolation between pillars |
|---|---|---|
| `SURVIVAL_PROBABILITY` | $Q(t_i)$ | linear in $Q$ |
| `DEFAULT_SPREAD` | $s(t_i) = -\ln Q(t_i)/t_i$ | linear in $s$ |
| `FORWARD_DEFAULT_SPREAD` | $\lambda_i$ on segment $(t_{i-1}, t_i]$ | piecewise-constant in $\lambda$ |

All three are equivalent in information content — given pillar dates and survival probabilities at those dates, the values of the other two can always be recovered. They differ only off-pillar, in *how* the curve is interpolated.

## 2. CDS Leg Decomposition (Risk-Neutral)

A vanilla single-name CDS exchanges:

- **Premium leg** (paid by the protection buyer): a coupon $s\cdot\alpha_i$ on each accrual end date $T_i$, conditional on no default; plus the **accrued premium** if default occurs mid-period.
- **Protection leg** (paid by the protection seller): a contingent payment of $1-R$ at the default time $\tau$ if $\tau$ falls within the contract life.

Under the risk-neutral measure with deterministic intensity, period-by-period the legs become integrals over $\lambda(t)\,Q(t)\,dt$:

$$
\text{Premium running PV} \;=\; s \sum_i \alpha_i \, P(T_i^{\text{pay}})\, Q(T_i)
$$

$$
\text{Premium accrual-on-default PV} \;=\; s \sum_i \int_{T_{i-1}}^{T_i} \alpha(T_{i-1}, t)\, P(t)\, \lambda(t)\,Q(t)\,dt
$$

$$
\text{Protection PV} \;=\; (1-R) \sum_i \int_{T_{i-1}}^{T_i} P(t)\, \lambda(t)\,Q(t)\,dt
$$

where $P(\cdot)$ is the (risk-free) discount factor. The integrals are evaluated period-by-period using the **midpoint discount factor** approximation, and the integrand $\lambda(t)Q(t)\,dt$ is replaced by its accrued increment $-dQ = Q(T_{i-1}) - Q(T_i)$:

$$
\int_{T_{i-1}}^{T_i} P(t)\,\lambda(t)\,Q(t)\,dt \;\approx\; \tfrac{1}{2}\big(P(T_{i-1}) + P(T_i)\big)\,\big(Q(T_{i-1}) - Q(T_i)\big)
$$

The accrued day-count is replaced by $\alpha_i/2$ — its midpoint over the period. This yields a closed-form expression that is exact in the continuous-time limit and consistent with standard market practice.

The **par spread** $s^*$ is the value that makes the CDS NPV zero at inception:

$$
s^* \;=\; \frac{\text{Protection PV}}{\text{RPV01}}
\qquad\text{where}\qquad
\text{RPV01} \;=\; \frac{\text{Premium PV}}{s}
$$

## 3. Building Block: `CdsQuote`

A `CdsQuote` bundles the market spread with all the conventions needed to generate the premium-leg schedule. The maturity date and the schedule are resolved **lazily** from the `reference_date` at bootstrap time, so the same quote object can be re-used across reference dates.

In [ ]:
example_quote = CdsQuote(
    spread=0.0125,                                    # 125 bps
    tenor="5Y",
    spot_lag=0,
    pay_frequency=Frequency.QUARTERLY,                # standard CDS coupon frequency
    calendar=USD,
    business_day_convention=MF,
    day_count_convention=ACT360,
    stub_type=StubType.SHORT_FRONT,                   # IMM/standard CDS convention
)

print(f"Spread:           {example_quote.quote_value():.4%}")
print(f"Tenor:            {example_quote.tenor}")
print(f"Start date:       {example_quote.start_date(REF)}")
print(f"Maturity date:    {example_quote.maturity_date(REF)}")

schedule_5y = example_quote.schedule(REF)
print(f"\nSchedule has {len(schedule_5y)} accrual periods.")
print(f"{'#':<3} {'accrual_start':<14} {'accrual_end':<14} {'pay_date':<14} {'dcf':>8}")
print("-" * 56)
for i, p in enumerate(schedule_5y[:4], 1):
    print(f"{i:<3} {str(p.accrual_start):<14} {str(p.accrual_end):<14} {str(p.pay_date):<14} {p.dcf:>8.6f}")
print("...")
for i, p in enumerate(schedule_5y[-2:], len(schedule_5y) - 1):
    print(f"{i:<3} {str(p.accrual_start):<14} {str(p.accrual_end):<14} {str(p.pay_date):<14} {p.dcf:>8.6f}")

## 4. The `CreditCurve` Object

`CreditCurve` is the container that the bootstrapper produces. It exposes:

- `non_default_probability(d)` — the survival probability $Q(d)$.
- `default_probability(d)` — its complement $1-Q(d)$.
- `default_spread(d)` — the cumulative-equivalent spread $s(d) = -\ln Q(d)/t(d)$.
- `forward_default_spread(start, end)` — the average forward hazard $-\ln(Q_{\text{end}}/Q_{\text{start}})/\Delta t$.

The interpolation choice does not change pillar values; it only changes how the curve fills in between pillars. To make this tangible, we instantiate three curves with **identical pillar survival probabilities** but different `InterpolationVariable` settings, and look at the off-pillar shapes.

In [ ]:
# Common pillars. Survival probabilities chosen by hand — same Q at every pillar across the
# three curves so the only difference is the off-pillar interpolation rule.
pillar_dates = [date(2025, 1, 2), date(2027, 1, 2), date(2029, 1, 2), date(2031, 1, 2)]
pillar_q     = [0.985, 0.948, 0.892, 0.815]
pillar_t     = [day_count_fraction(REF, d, ACT365) for d in pillar_dates]

# Convert to the values each interpolation variable expects.
pillar_s     = [-math.log(q) / t for q, t in zip(pillar_q, pillar_t)]   # cumulative spread
pillar_lam = []                                                         # piecewise-constant forward hazard
prev_q, prev_t = 1.0, 0.0
for q, t in zip(pillar_q, pillar_t):
    pillar_lam.append(-math.log(q / prev_q) / (t - prev_t))
    prev_q, prev_t = q, t

curve_q   = CreditCurve(REF, pillar_dates, pillar_q,   InterpolationVariable.SURVIVAL_PROBABILITY,    ACT365)
curve_s   = CreditCurve(REF, pillar_dates, pillar_s,   InterpolationVariable.DEFAULT_SPREAD,          ACT365)
curve_lam = CreditCurve(REF, pillar_dates, pillar_lam, InterpolationVariable.FORWARD_DEFAULT_SPREAD,  ACT365)

print("Pillar values fed in (each curve sees the variable it expects):\n")
print(f"{'Pillar':<12} {'t':>8} {'Q':>10} {'s (cum)':>10} {'lambda (fwd)':>14}")
print("-" * 56)
for d, t, q, s, lam in zip(pillar_dates, pillar_t, pillar_q, pillar_s, pillar_lam):
    print(f"{str(d):<12} {t:>8.4f} {q:>10.6f} {s:>10.4%} {lam:>14.4%}")

In [ ]:
# Sample each curve daily for 7 years and plot Q(t).
from datetime import timedelta

sample_days = list(range(1, 365 * 7 + 1, 7))
sample_dates = [REF + timedelta(days=d) for d in sample_days]
t_axis = [d / 365.0 for d in sample_days]

q_q   = [curve_q.non_default_probability(d)   for d in sample_dates]
q_s   = [curve_s.non_default_probability(d)   for d in sample_dates]
q_lam = [curve_lam.non_default_probability(d) for d in sample_dates]

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(t_axis, q_q,   color='steelblue',   linewidth=2, label='SURVIVAL_PROBABILITY (linear in Q)')
ax.plot(t_axis, q_s,   color='tomato',      linewidth=2, label='DEFAULT_SPREAD (linear in s)')
ax.plot(t_axis, q_lam, color='mediumpurple', linewidth=2, label='FORWARD_DEFAULT_SPREAD (piecewise-constant lambda)')
ax.scatter(pillar_t, pillar_q, color='black', s=40, zorder=5, label='pillar Q')
ax.set_xlabel('Years')
ax.set_ylabel('Survival probability Q(t)')
ax.set_title('Same pillar Q, three interpolation rules')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Off-pillar all three curves agree at the pillar dates but disagree in between. The `FORWARD_DEFAULT_SPREAD` variable is what most ISDA-style implementations use — it gives a piecewise-exponential $Q(t)$ which corresponds to a piecewise-constant hazard rate, the simplest model consistent with the deterministic-intensity setup.

## 5. The Pricer: `SingleNameCDS`

The bootstrapper drives an NPV root-finder, and the NPV comes from `SingleNameCDS`. The pricer needs **both** a `ZeroCurve` (for $P(t)$) and a `CreditCurve` (for $Q(t)$). Below we build a flat 3% USD discount curve and re-use the 5Y CDS quote with the `FORWARD_DEFAULT_SPREAD` curve from above.

We then verify the building blocks individually — protection leg, premium leg, RPV01, par spread — and confirm that NPV is zero when the running spread equals the par spread.

In [ ]:
# Flat 3% continuously-compounded USD discount curve out to 12Y.
zero_curve = ZeroCurve(
    reference_date=REF,
    pillar_dates=[date(2036, 1, 2)],
    rates=[0.03],
    day_count_convention=ACT365,
)

cds_5y = SingleNameCDS.from_quote(
    quote=example_quote,                # 125 bps, 5Y
    reference_date=REF,
    recovery_rate=RECOVERY,
    zero_curve=zero_curve,
    credit_curve=curve_lam,
    side=CdsSide.BUYER,
)

print("Leg-by-leg PV breakdown for the 5Y CDS @ 125 bps:")
print(f"  Protection leg PV               = {cds_5y.protection_leg_pv():>+12.8f}")
print(f"  Premium leg running PV          = {cds_5y.premium_leg_running_pv():>+12.8f}")
print(f"  Premium accrual-on-default PV   = {cds_5y.accrual_on_default_pv():>+12.8f}")
print(f"  Premium leg PV (total)          = {cds_5y.premium_leg_pv():>+12.8f}")
print(f"  RPV01 (risky annuity)           = {cds_5y.rpv01():>+12.8f}")
print(f"  Par spread                      = {cds_5y.par_spread():>+12.4%}")
print(f"  NPV (buyer)                     = {cds_5y.npv():>+12.8f}")

In [ ]:
# Sanity check: NPV must vanish when the running spread equals par.
par = cds_5y.par_spread()
cds_at_par = SingleNameCDS(
    reference_date=REF,
    periods=example_quote.schedule(REF),
    spread=par,
    recovery_rate=RECOVERY,
    zero_curve=zero_curve,
    credit_curve=curve_lam,
)
print(f"NPV at par spread ({par:.6%}): {cds_at_par.npv():.2e}")
print("This is the root system the bootstrapper solves at every pillar.")

## 6. Bootstrapping with `CreditCurveBootstrapper`

We now flip the construction around: we **observe market par spreads** at standard tenors and we want a credit curve that re-prices each one to zero NPV. The bootstrapper takes:

- a list of `CdsQuote` objects (sorted internally by maturity),
- the discount `ZeroCurve` (built externally — see notebook 04),
- a constant recovery rate,
- the `InterpolationVariable` to solve for,
- and the `BootstrapMode`.

We use a five-tenor USD term structure of CDS spreads typical of an investment-grade name.

In [ ]:
# Five-tenor IG-style CDS term structure.
spreads_bps = {"1Y": 80, "3Y": 110, "5Y": 130, "7Y": 145, "10Y": 155}

def make_quote(tenor: str, bps: float) -> CdsQuote:
    return CdsQuote(
        spread=bps / 10_000.0,
        tenor=tenor,
        spot_lag=0,
        pay_frequency=Frequency.QUARTERLY,
        calendar=USD,
        business_day_convention=MF,
        day_count_convention=ACT360,
        stub_type=StubType.SHORT_FRONT,
    )

market_quotes = [make_quote(t, bps) for t, bps in spreads_bps.items()]

print(f"{'Tenor':<5} {'Spread (bps)':>14} {'Maturity':>12}")
print("-" * 35)
for t, q in zip(spreads_bps, market_quotes):
    print(f"{t:<5} {q.quote_value()*10_000:>14.2f} {str(q.maturity_date(REF)):>12}")

### 6a. Sequential mode — pillar by pillar

`BootstrapMode.SEQUENTIAL` solves one pillar at a time. The curve is built incrementally: the $i$-th iteration treats the values at pillars $1, \dots, i-1$ as fixed and uses scalar Newton-Raphson to find the value at pillar $i$ that makes the $i$-th CDS price to zero NPV. This is well-conditioned because the $i$-th CDS only depends on the $i$-th pillar value through its effect on $Q(t)$ for $t \in (t_{i-1}, t_i]$.

We trace the build-up explicitly: bootstrap with the first $k$ quotes only, for $k = 1, 2, \dots, 5$, and watch the partial curve grow.

In [ ]:
for k in range(1, len(market_quotes) + 1):
    partial = CreditCurveBootstrapper(
        reference_date=REF,
        quotes=market_quotes[:k],
        zero_curve=zero_curve,
        recovery_rate=RECOVERY,
        interpolation_variable=InterpolationVariable.FORWARD_DEFAULT_SPREAD,
        mode=BootstrapMode.SEQUENTIAL,
    ).bootstrap()
    last = partial.pillar_dates[-1]
    print(f"After pillar {k} ({list(spreads_bps)[k-1]:>3}):  Q({last}) = {partial.non_default_probability(last):.6f}   "
          f"forward hazard segment {k} = {partial.pillar_values[-1]:.4%}")

In [ ]:
# Final sequential bootstrap and a tabular summary.
credit_seq = CreditCurveBootstrapper(
    reference_date=REF,
    quotes=market_quotes,
    zero_curve=zero_curve,
    recovery_rate=RECOVERY,
    interpolation_variable=InterpolationVariable.FORWARD_DEFAULT_SPREAD,
    mode=BootstrapMode.SEQUENTIAL,
).bootstrap()

print(credit_seq.summary())

### 6b. Global mode — all pillars at once

`BootstrapMode.GLOBAL` solves the full vector system

$$
F(\lambda_1, \dots, \lambda_n) \;=\; \big(\text{NPV}_1(\lambda), \dots, \text{NPV}_n(\lambda)\big) \;=\; \mathbf{0}
$$

in one shot using **multivariate Newton-Raphson** with a finite-difference Jacobian. At each iteration the linear system $J\,\Delta\lambda = -F(\lambda)$ is solved by Gaussian elimination with partial pivoting.

When the system is exactly determined and the interpolation variable is causal (each pillar only affects $Q$ to the right of itself), `GLOBAL` and `SEQUENTIAL` converge to the same solution. `GLOBAL` is more robust to coupling — useful when the interpolation rule lets a later pillar affect $Q$ at an earlier date, e.g. with non-causal smoothing — and it converges all residuals at once rather than serially.

In [ ]:
credit_glob = CreditCurveBootstrapper(
    reference_date=REF,
    quotes=market_quotes,
    zero_curve=zero_curve,
    recovery_rate=RECOVERY,
    interpolation_variable=InterpolationVariable.FORWARD_DEFAULT_SPREAD,
    mode=BootstrapMode.GLOBAL,
).bootstrap()

print("Pillar-by-pillar agreement between SEQUENTIAL and GLOBAL:\n")
print(f"{'Pillar':<12} {'Q (sequential)':>17} {'Q (global)':>14} {'|diff|':>14}")
print("-" * 60)
for d in credit_seq.pillar_dates:
    q_seq  = credit_seq.non_default_probability(d)
    q_glob = credit_glob.non_default_probability(d)
    print(f"{str(d):<12} {q_seq:>17.10f} {q_glob:>14.10f} {abs(q_seq-q_glob):>14.2e}")

## 7. Round-Trip Verification

A correctly bootstrapped curve must re-price every input quote to zero NPV. We re-construct each `SingleNameCDS` on the calibrated curve and report the residual.

In [ ]:
print(f"{'Tenor':<5} {'Spread (bps)':>14} {'NPV (sequential)':>20} {'NPV (global)':>18}")
print("-" * 60)
for tenor, q in zip(spreads_bps, market_quotes):
    npv_seq  = SingleNameCDS.from_quote(q, REF, RECOVERY, zero_curve, credit_seq).npv()
    npv_glob = SingleNameCDS.from_quote(q, REF, RECOVERY, zero_curve, credit_glob).npv()
    print(f"{tenor:<5} {q.quote_value()*10_000:>14.2f} {npv_seq:>20.2e} {npv_glob:>18.2e}")

## 8. Side-by-Side: Three Interpolation Variables on the Same Quotes

Bootstrapping with each `InterpolationVariable` on the same five market quotes: all three calibrate to zero NPV at the pillars (round-trip is exact by construction), but they differ off-pillar. The differences show up most clearly in the implied forward hazard rate $\lambda(t)$, which is *piecewise-constant* under `FORWARD_DEFAULT_SPREAD` and *smoothly varying* under the other two.

In [ ]:
variables = {
    "FORWARD_DEFAULT_SPREAD":  InterpolationVariable.FORWARD_DEFAULT_SPREAD,
    "DEFAULT_SPREAD":          InterpolationVariable.DEFAULT_SPREAD,
    "SURVIVAL_PROBABILITY":    InterpolationVariable.SURVIVAL_PROBABILITY,
}

calibrated = {}
for name, var in variables.items():
    calibrated[name] = CreditCurveBootstrapper(
        reference_date=REF,
        quotes=market_quotes,
        zero_curve=zero_curve,
        recovery_rate=RECOVERY,
        interpolation_variable=var,
        mode=BootstrapMode.SEQUENTIAL,
    ).bootstrap()

# Round-trip residuals — must all be small.
print("Round-trip max |NPV| per interpolation variable:")
for name, curve in calibrated.items():
    max_resid = max(abs(SingleNameCDS.from_quote(q, REF, RECOVERY, zero_curve, curve).npv()) for q in market_quotes)
    print(f"  {name:<24}  {max_resid:.2e}")

In [ ]:
from datetime import timedelta

sample_days_long = list(range(7, 365 * 10 + 1, 7))
sample_dates_long = [REF + timedelta(days=d) for d in sample_days_long]
t_axis_long = [d / 365.0 for d in sample_days_long]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
colours = {'FORWARD_DEFAULT_SPREAD': 'mediumpurple', 'DEFAULT_SPREAD': 'tomato', 'SURVIVAL_PROBABILITY': 'steelblue'}

# Q(t)
ax = axes[0]
for name, curve in calibrated.items():
    ys = [curve.non_default_probability(d) for d in sample_dates_long]
    ax.plot(t_axis_long, ys, color=colours[name], linewidth=1.8, label=name)
ax.set_title('Survival probability $Q(t)$')
ax.set_xlabel('Years'); ax.set_ylabel('Q(t)')
ax.grid(True, alpha=0.3); ax.legend(fontsize=8)

# Cumulative default spread s(t)
ax = axes[1]
for name, curve in calibrated.items():
    ys = [curve.default_spread(d) * 10_000 for d in sample_dates_long]
    ax.plot(t_axis_long, ys, color=colours[name], linewidth=1.8, label=name)
ax.set_title('Cumulative default spread $s(t)$ (bps)')
ax.set_xlabel('Years'); ax.set_ylabel('s(t) (bps)')
ax.grid(True, alpha=0.3); ax.legend(fontsize=8)

# Forward hazard, sampled over rolling 1M windows
ax = axes[2]
fwd_starts = [REF + timedelta(days=d) for d in range(7, 365 * 10, 30)]
fwd_ends   = [s + timedelta(days=30) for s in fwd_starts]
fwd_t      = [(s - REF).days / 365.0 for s in fwd_starts]
for name, curve in calibrated.items():
    ys = [curve.forward_default_spread(s, e) * 10_000 for s, e in zip(fwd_starts, fwd_ends)]
    ax.plot(fwd_t, ys, color=colours[name], linewidth=1.8, label=name)
ax.set_title('1M rolling forward hazard (bps)')
ax.set_xlabel('Years'); ax.set_ylabel('forward $\\lambda$ (bps)')
ax.grid(True, alpha=0.3); ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

All three calibrations match the same five market quotes exactly at the pillar dates — the differences are entirely off-pillar interpolation. The forward-hazard chart in the right panel is the most striking: under `FORWARD_DEFAULT_SPREAD` the hazard is flat between pillars and jumps at each one, while the other two parameterisations smooth the hazard through the pillars.

Choosing between them is a modelling decision:

- `FORWARD_DEFAULT_SPREAD` — ISDA-style; minimal assumptions; the hazard is undefined at pillar dates but constant in between.
- `DEFAULT_SPREAD` — natural for risk-management contexts where one quotes implied default rates per tenor and wants a smooth $s(t)$.
- `SURVIVAL_PROBABILITY` — direct, but linear interpolation of $Q$ is generally less smooth than the alternatives and can give counter-intuitive forward hazards near the pillars.

## 9. Putting It All Together

The full pipeline used in production is:

1. Build a discount `ZeroCurve` from rate market data (notebook 04).
2. Collect CDS spreads at standard tenors and wrap them as `CdsQuote` objects.
3. Choose a `recovery_rate` (often 40% for senior unsecured corporates), an `InterpolationVariable`, and a `BootstrapMode`.
4. Run `CreditCurveBootstrapper.bootstrap()` to produce a calibrated `CreditCurve`.
5. Use the calibrated `(ZeroCurve, CreditCurve)` pair inside `SingleNameCDS` to price related instruments — off-the-run CDS, CDS options, structured credit baskets — consistently with the market term structure of spreads.

The same `SingleNameCDS` pricer drives the bootstrapper's NPV objective and any downstream pricing query, so calibration and valuation are guaranteed to be self-consistent.